## Pyspark Connection


Import library


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, when, sum as _sum, round, year, month


In [2]:
# Configure spark session
spark = SparkSession\
    .builder\
    .master('local[*]')\
    .appName('myapp')\
    .config('spark.jars.packages', 'mysql:mysql-connector-java:5.1.44')\
    .getOrCreate()

Extract Data from Mysql

In [5]:
df_ecommerce_db = spark.read.format('jdbc').\
            option('url', 'jdbc:mysql://localhost:3306/ecommerce_db').\
            option('driver', 'com.mysql.jdbc.Driver').\
            option('user', 'testuser').\
            option('password', '1234').\
            option('dbtable', 'ecommerce_orders').\
            load()

In [6]:
df_ecommerce_db.show(5)

+-------------------+----------+--------------------+----------+-------------+-------------+-------+---------------+-------------+----+----------+--------------+--------+--------+------+-----------+-----------+-----------+-------+---+------------+
|           order_id|order_date|              status|fulfilment|sales_channel|service_level|  style|            sku|     category|size|      asin|courier_status|quantity|currency|amount|       city|      state|postal_code|country|b2b|fulfilled_by|
+-------------------+----------+--------------------+----------+-------------+-------------+-------+---------------+-------------+----+----------+--------------+--------+--------+------+-----------+-----------+-----------+-------+---+------------+
|405-8078784-5731545|2022-04-30|           Cancelled|  Merchant|    Amazon.in|     Standard| SET389| SET389-KR-NP-S|          Set|   S|B09KXVBD7Z|          NULL|       0|     INR|647.62|     MUMBAI|MAHARASHTRA|     400081|     IN|  0|   Easy Ship|
|171-919

### 1. Transformation (The "T" in ETL)

Drop column that aren't needed

In [ ]:
# Select the columns we need
# Keep the data focused on the assignment output
df_refined = df_ecommerce_db.select(
    col("order_id"),
    col("order_date"),
    col("category"),
    col("amount"),
    col("quantity"),
    col("state"),
    col("status"),
    col("b2b")
)

# Convert the date and add simple time features
df_refined = df_refined.withColumn("order_date", to_date(col("order_date"), "MM-dd-yy"))
df_refined = df_refined.withColumn("year", year(col("order_date")))
df_refined = df_refined.withColumn("month", month(col("order_date")))

# Remove invalid rows
df_refined = df_refined.filter(
    col("amount").isNotNull()
    & col("quantity").isNotNull()
    & (col("amount") > 0)
    & (col("quantity") > 0)
)

# Show the cleaned structure
df_refined.printSchema()
df_refined.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- state: string (nullable = true)
 |-- status: string (nullable = true)
 |-- b2b: string (nullable = true)

+-------------------+----------+-------------+------+--------+-----------+--------------------+---+
|           order_id|order_date|     category|amount|quantity|      state|              status|b2b|
+-------------------+----------+-------------+------+--------+-----------+--------------------+---+
|405-8078784-5731545|2022-04-30|          Set|647.62|       0|MAHARASHTRA|           Cancelled|  0|
|171-9198151-1101146|2022-04-30|        kurta| 406.0|       1|  KARNATAKA|Shipped - Deliver...|  0|
|404-0687676-7273146|2022-04-30|        kurta| 329.0|       1|MAHARASHTRA|             Shipped|  1|
|403-9615377-8133951|2022-04-30|Western Dress|753.33|       0| PUDUCHERRY|           C

### Exploratory Data Analysis (EDA)

In [ ]:
# 1. Check for missing values in the cleaned columns
print("Null Value Report:")
df_refined.select([count(when(col(c).isNull(), c)).alias(c) for c in df_refined.columns]).show()

# 2. Get basic statistics for numerical columns
print("Statistical Summary:")
df_refined.select("amount", "quantity").describe().show()

# 3. Check the distribution of order status
print("Order Status Distribution:")
df_refined.groupBy("status").count().orderBy("count", ascending=False).show()

# 4. Count orders per category
print("Orders Per Category:")
df_refined.groupBy("category").count().orderBy("count", ascending=False).show()

Null Value Report:
+--------+----------+--------+------+--------+-----+------+---+
|order_id|order_date|category|amount|quantity|state|status|b2b|
+--------+----------+--------+------+--------+-----+------+---+
|       0|         0|       0|  7795|       0|   33|     0|  0|
+--------+----------+--------+------+--------+-----+------+---+

Statistical Summary:
+-------+-----------------+------------------+
|summary|           amount|          quantity|
+-------+-----------------+------------------+
|  count|           121180|            128975|
|   mean|648.5614646813405|0.9044310912967629|
| stddev|281.2116871742239|0.3133535856501436|
|    min|              0.0|                 0|
|    max|           5584.0|                15|
+-------+-----------------+------------------+

Order Status Distribution:
+--------------------+-----+
|              status|count|
+--------------------+-----+
|             Shipped|77804|
|Shipped - Deliver...|28769|
|           Cancelled|18332|
|Shipped - Ret

### Data Aggregation

In [ ]:
# Aggregate sales by category
df_category_summary = df_refined.groupBy("category") \
    .agg(
        _sum("quantity").alias("total_items_sold"),
        round(_sum("amount"), 2).alias("total_revenue")
    ) \
    .orderBy("total_revenue", ascending=False)

print("Business Insight: Sales by Category")
df_category_summary.show()

# Revenue by state
df_state_summary = df_refined.groupBy("state") \
    .agg(
        round(_sum("amount"), 2).alias("total_revenue"),
        _sum("quantity").alias("total_quantity")
    ) \
    .orderBy("total_revenue", ascending=False)

print("Business Insight: Revenue by State")
df_state_summary.show()

# Monthly sales trend
df_monthly_summary = df_refined.groupBy("year", "month") \
    .agg(
        round(_sum("amount"), 2).alias("total_revenue"),
        _sum("quantity").alias("total_quantity")
    ) \
    .orderBy("year", "month")

print("Business Insight: Monthly Sales Trend")
df_monthly_summary.show()

# Optional B2B comparison
df_b2b_summary = df_refined.groupBy("b2b") \
    .agg(round(_sum("amount"), 2).alias("total_revenue")) \
    .orderBy("total_revenue", ascending=False)

print("Business Insight: B2B vs Non-B2B Revenue")
df_b2b_summary.show()

Business Insight: Sales by Category
+-------------+----------------+-------------+
|     category|total_items_sold|total_revenue|
+-------------+----------------+-------------+
|          Set|           45289|3.920412402E7|
|        kurta|           45045| 2.12995467E7|
|Western Dress|           13943|1.121607269E7|
|          Top|            9903|    5347792.3|
| Ethnic Dress|            1053|    791217.66|
|       Blouse|             863|    458408.18|
|       Bottom|             398|    150667.98|
|        Saree|             152|    123933.76|
|      Dupatta|               3|        915.0|
+-------------+----------------+-------------+



### Loading (The Final "L" in ETL)

In [ ]:
# Save the category summary into MySQL
df_category_summary.write \
    .format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3306/ecommerce_db") \
    .option("driver", "com.mysql.jdbc.Driver") \
    .option("user", "testuser") \
    .option("password", "1234") \
    .option("dbtable", "category_sales_summary") \
    .mode("overwrite") \
    .save()

# Save one more useful summary table
df_state_summary.write \
    .format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3306/ecommerce_db") \
    .option("driver", "com.mysql.jdbc.Driver") \
    .option("user", "testuser") \
    .option("password", "1234") \
    .option("dbtable", "state_sales_summary") \
    .mode("overwrite") \
    .save()

print("Assignment complete. Summary tables saved to MySQL.")

Assignment Complete! Summary table 'category_sales_summary' saved to MySQL.
